In [1]:
!uv pip install altair

Audited 1 package in 22ms


In [2]:
import polars as pl
from polars import col


pl.Config.set_tbl_rows(1000)      # Allow rendering up to 1000 rows inline
pl.Config.set_tbl_cols(20)        # Allow rendering up to 20 columns wide
pl.Config.set_fmt_str_lengths(50) # Prevent text strings from getting cropped

market_response_path = "/Users/oceansxyz/alphanode-metal/logs/market/market_20260614_105506.jsonl"

#### Load JSONL

In [3]:
#Raw binance reponse for market web socket - pretty much a static
book_ticker_schema = pl.Struct([
    pl.Field("ts", pl.Int64),
    pl.Field("frame", pl.Struct([
        pl.Field("u", pl.Int64),
        pl.Field("s", pl.String),
        pl.Field("b", pl.String),
        pl.Field("B", pl.String),
        pl.Field("a", pl.String),
        pl.Field("A", pl.String),
    ])),
])

with open(market_response_path) as f:
    metadata_json = f.readline().strip()
    print("Metadata Raw Line:", metadata_json)
    df_market = (
        pl.read_csv(
            f,
            has_header=False,
            new_columns=["json_str"],
            separator="\n",
            infer_schema=False,
        )
        .select(pl.col("json_str").str.json_decode(dtype=book_ticker_schema))
        .unnest("json_str")
        .unnest("frame")
        .with_columns([
            pl.col("u").cast(pl.Int64),
            pl.col("s"),
            pl.col("b").cast(pl.Float64),
            pl.col("B").cast(pl.Float64),
            pl.col("a").cast(pl.Float64),
            pl.col("A").cast(pl.Float64),
        ])
    )


# Select relevant cols for alpha calulation with identifiable file names
market_rates = df_market.select(
    col("ts"), col("s").alias("symbol"), col("b").alias("bid"), col("a").alias("ask")
)

# duration of web socket listen
min_ts = market_rates.min().select("ts").item()
max_ts = market_rates.max().select("ts").item()
duration_m =(max_ts-min_ts)/60000


print(f"Duration of listen : {duration_m} mins")
print(market_rates.group_by("symbol").len())

Metadata Raw Line: {"ts":1781434506715,"frame":{"result":null,"id":1}}
Duration of listen : 9.4146 mins
shape: (3, 2)
┌──────────┬──────┐
│ symbol   ┆ len  │
│ ---      ┆ ---  │
│ str      ┆ u32  │
╞══════════╪══════╡
│ BTCEURI  ┆ 115  │
│ EURIUSDT ┆ 8    │
│ BTCUSDT  ┆ 2966 │
└──────────┴──────┘


In [4]:
## Split dataframe symbolwise symbol is an orderbook
symbol_0 = market_rates.filter(col("symbol")=="BTCUSDT").select(["ts", col("bid").alias("BTCUSDT_bid"), col("ask").alias("BTCUSDT_ask")])
symbol_1 = market_rates.filter(col("symbol")=="BTCEURI").select(["ts", col("bid").alias("BTCEURI_bid"), col("ask").alias("BTCEURI_ask")])
symbol_2 = market_rates.filter(col("symbol")=="EURIUSDT").select(["ts", col("bid").alias("EURIUSDT_bid"), col("ask").alias("EURIUSDT_ask")])

In [5]:
# Forward fill missing values by copying the last known valid price down 
# for all generated ticker metrics simultaneously, skipping the timestamp.
market_rates_widened = symbol_0.join(symbol_1, on="ts", how="full", coalesce=True)\
    .join(symbol_2, on="ts", how="full", coalesce=True)\
    .sort("ts")\
    .select([
        col("ts"),
        col("*").exclude("ts").fill_null(strategy="forward")
    ]).drop_nulls()

market_rates_widened.head()

ts,BTCUSDT_bid,BTCUSDT_ask,BTCEURI_bid,BTCEURI_ask,EURIUSDT_bid,EURIUSDT_ask
i64,f64,f64,f64,f64,f64,f64
1781434666277,64531.44,64531.45,55825.34,55847.46,1.1556,1.1559
1781434666590,64531.44,64531.45,55825.34,55847.46,1.1556,1.1559
1781434666781,64531.44,64531.45,55825.34,55847.46,1.1556,1.1559
1781434667338,64531.44,64531.45,55825.34,55847.46,1.1556,1.1559
1781434667587,64531.44,64531.45,55825.34,55847.46,1.1556,1.1559


##### Notation & Model

NODE_0 = USDT NODE_1 = BTC NODE_2 = EURI  

EDGE_0 = BTCUSDT  EDGE_1 = BTCEURI  EDGE_2 = EURIUSDT  

Clockwise Walk  0→1  →1→2  →2→0  

BUY  BTCUSDT  @ASK  === 0→1  
SELL BTCEURI  @BID  === 1→2  
SELL EURIUSDT @BID  === 2→0


#### Clockwise Walk  0→1  →1→2  →2→0

In [8]:
clockwise_walk = market_rates_widened.select("ts", col("BTCUSDT_ask"), "BTCEURI_bid", "EURIUSDT_bid")\
    .unique(subset=["BTCUSDT_ask", "BTCEURI_bid", "EURIUSDT_bid"])\
    .with_columns((col("BTCEURI_bid")*col("EURIUSDT_bid")).alias("BTCEURIUSDT_bid"))\
    .with_columns((col("BTCEURIUSDT_bid")-col("BTCUSDT_ask")).abs().alias("alpha"))

clockwise_walk.sort("alpha", descending=True)



ts,BTCUSDT_ask,BTCEURI_bid,EURIUSDT_bid,BTCEURIUSDT_bid,alpha
i64,f64,f64,f64,f64,f64
1781435059590,64492.0,55767.39,1.1556,64444.795884,47.204116
1781435059589,64491.99,55767.39,1.1556,64444.795884,47.194116
1781435059589,64491.36,55767.39,1.1556,64444.795884,46.564116
1781435059589,64490.82,55767.39,1.1556,64444.795884,46.024116
1781435059589,64490.53,55767.39,1.1556,64444.795884,45.734116
1781435059589,64490.51,55767.39,1.1556,64444.795884,45.714116
1781435059589,64490.49,55767.39,1.1556,64444.795884,45.694116
1781435059589,64490.39,55767.39,1.1556,64444.795884,45.594116
1781434871598,64517.9,55791.26,1.1556,64472.380056,45.519944


In [7]:
clockwise_walk.plot.line(x="ts", y="alpha")

alt.Chart(...)